# Map of all compounds in the input dataset

PubChem, HMDB, NIST.

In [ ]:
INPUT_FILE = "Compounds.xlsx"
INPUT_DIRECTORY = "../data/"
OUTPUT_DIRECTORY = "../results/"

In [ ]:
import pandas as pd
import os
import requests
import nistchempy as nist

In [ ]:
df = pd.read_excel(os.path.join(INPUT_DIRECTORY, INPUT_FILE), header=None)
df.head()

df.rename(columns={0: "Input Name"}, inplace=True)

df.insert(1, "Common/IUPAC Name", None)
df.insert(2, "InChIKey", None)
df.insert(3, "CID", None)
df.insert(4, "RI Polar (NIST)", None)
df.insert(5, "HMDB ID", None)
df.insert(6, "Ontology Links", None)
df.insert(7, "Pathway", None)
df.insert(8, "Disease Association", None)
df.insert(2, "InChI", None)
df.insert(10, "SMILES", None)
df.insert(11, "Taxonomy", None)
df.insert(12, "Is Terpene", None)

df.head(5)

## PubChem API

By Input Name we retrieve CID, inchikey, InChi, InChiKey, IUPAC Name and all synonyms.

In [ ]:
PUBCHEM_API_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound"

compound_synonyms = pd.DataFrame()
compound_synonyms.insert(0, "Input Name", None)
compound_synonyms.insert(1, "CID", None)

for index, row in df.iterrows():
    compound_name = row["Input Name"]
    pubchem_synonyms_result = requests.get(f"{PUBCHEM_API_URL}/name/{compound_name}/synonyms/JSON").json()
    if "InformationList" not in pubchem_synonyms_result or len(pubchem_synonyms_result["InformationList"]["Information"]) == 0:
        print(f"No result in PubChem found for {index}: {compound_name}")
        continue
    synonyms = pubchem_synonyms_result["InformationList"]["Information"][0]["Synonym"]
    cid = pubchem_synonyms_result["InformationList"]["Information"][0]["CID"]
    compound_synonyms.at[index, "Input Name"] = compound_name
    compound_synonyms.at[index, "CID"] = cid
    df.at[index, "CID"] = cid
    for i, s in enumerate(synonyms):
        if i > 20: # limit to 20 synonyms to avoid too many columns
            break
        if i >= len(compound_synonyms.columns) - 2: # if we need more columns, add them
            compound_synonyms.insert(len(compound_synonyms.columns), f"Synonym {i}", None)
        compound_synonyms.at[index, f"Synonym {i}"] = s

In [ ]:
for index, row in df.iterrows():
    cid = row["CID"]
    pubchem_result = requests.get(f"{PUBCHEM_API_URL}/cid/{cid}/property/Title,SMILES,InChI,InChIKey,IUPACName/JSON").json()
    if "PropertyTable" not in pubchem_result or len(pubchem_result["PropertyTable"]["Properties"]) == 0:
        print(f"No result in PubChem found for CID {row['Input Name']}")
        continue
    props = pubchem_result["PropertyTable"]["Properties"][0]
    df.at[index, "Common/IUPAC Name"] = props["IUPACName"]
    df.at[index, "InChI"] = props["InChI"]
    df.at[index, "InChIKey"] = props["InChIKey"]
    df.at[index, "SMILES"] = props["SMILES"]

df.head(5)

In [ ]:
compound_synonyms.to_csv(os.path.join(OUTPUT_DIRECTORY, "compound_synonyms.csv"), index=False)
compound_synonyms.head()

## HMDB

Retrieving disease associations and pathways.

In [ ]:
import math
import sqlite3


HMDB_SQLITE_PATH = "../data/hmdb.sqlite"


def is_missing(x):
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    s = str(x).strip().lower()
    return s == "" or s in {"none", "nan", "null"}


def norm_text(x):
    if is_missing(x):
        return None
    return str(x).strip().lower()


conn = sqlite3.connect(HMDB_SQLITE_PATH)

conn.row_factory = sqlite3.Row

for index, row in df.iterrows():
    inchi = row.get("InChI")
    inchikey = row.get("InChIKey")
    smiles = row.get("SMILES")

    cursor = conn.execute(
        """
        SELECT hmdb_id
        FROM metabolites
        WHERE inchi_key = ?
           OR inchi = ?
           OR smiles = ?
        LIMIT 1
        """,
        (inchikey, inchi, smiles)
    )
    result = cursor.fetchone()

    if result is not None:
        df.at[index, "HMDB ID"] = result["hmdb_id"]
        print(f"matched by identifier: row {index} -> {result['hmdb_id']}")


for index, row in df.iterrows():
    if not is_missing(row.get("HMDB ID")):
        continue

    input_name = row.get("Input Name")
    if is_missing(input_name):
        print(f"not found for row {index}: empty Input Name")
        continue

    syn_rows = compound_synonyms.loc[
        compound_synonyms["Input Name"] == input_name
    ]

    if syn_rows.empty:
        print(f"not found for {input_name}: no synonyms row")
        continue

    found = False

    for _, syn_row in syn_rows.iterrows():
        for col in syn_row.index:
            if col == "Input Name":
                continue

            synonym = syn_row[col]
            if is_missing(synonym):
                continue

            synonym = str(synonym).strip()

            cursor = conn.execute(
                """
                SELECT hmdb_id, name, synonyms
                FROM metabolites
                WHERE lower(name) = lower(?)
                   OR lower(synonyms) LIKE lower(?)
                LIMIT 1
                """,
                (synonym, f"%{synonym}%")
            )
            result = cursor.fetchone()

            if result is not None:
                df.at[index, "HMDB ID"] = result["hmdb_id"]
                print(
                    f"matched by synonym: row {index} ({input_name}) "
                    f"using '{synonym}' -> {result['hmdb_id']}"
                )
                found = True
                break

        if found:
            break

    if not found:
        print(f"not found for {input_name}")

conn.close()

In [ ]:
# fill in all HMDB info
import json


conn = sqlite3.connect(HMDB_SQLITE_PATH)

# df.insert(9, "RI (HMDB)", None)
# df.insert(10, "Biomarker", None)

for index, row in df.iterrows():
    hmdb_id = row.get("HMDB ID")
    if is_missing(hmdb_id):
        continue

    cursor = conn.execute(
        """
        SELECT o.metabolic_processes_json, o.diseases_json, m.chemical_taxonomy_json, m.retention_indices_json, o.biomarker_for_json
        FROM metabolites m
        JOIN ontology o ON m.hmdb_id = o.hmdb_id
        WHERE m.hmdb_id = ?
        """,
        (hmdb_id,)
    )
    result = cursor.fetchone()


    if result is not None:
        # df.at[index, "Ontology Links"] = result["ontology_links"]
        df.at[index, "Pathway"] = result[0]  # metabolic_processes_json
        df.at[index, "Disease Association"] = result[1]  # diseases_json
        df.at[index, "Taxonomy"] = result[2]  # chemical_taxonomy_json
        df.at[index, "Is Terpene"] = "terpene" in result[2].lower() if result[2] else None
        df.at[index, "RI (HMDB)"] = result[3]  # retention_indices_json
        df.at[index, "Biomarker"] = result[4]  # biomarker_for_json

In [ ]:
df.to_csv(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.csv"), index=False)

# df[~df["Biomarker"].isna() & (df["Biomarker"] != "[]")]
# df[~df["Disease Association"].isna() & (df["Disease Association"] != "[]")]

## ChEBI

By SMILES we retrieve ontology relations and HMDB ID for the compound.

In [ ]:
# CHEBI_API_URL = "https://www.ebi.ac.uk/chebi/backend/api/public/"

# for id, row in df.iterrows():
#     inchi = df.at[id, "InChI"]
#     name = df.at[id, "Input Name"]
#     chebi_id_request = requests.get(f"{CHEBI_API_URL}es_search/?term={inchi}&page=1&size=15").json()

#     if chebi_id_request is None or "results" not in chebi_id_request or len(chebi_id_request["results"]) == 0:
#         print(f"No result in ChEBI found for [{id}] {name} ({inchi})")
#         continue

#     chebi_id = chebi_id_request["results"][0]["_source"]["chebi_accession"]

#     chebi_request = requests.get(f"{CHEBI_API_URL}compound/{chebi_id}?only_ontology_parents=false&only_ontology_children=false").json()
    
#     if chebi_request is None or "database_accessions" not in chebi_request or "MANUAL_X_REF" not in chebi_request["database_accessions"]:
#         print(f"No result in ChEBI found for [{id}] {name}")
#         continue

#     hmdb_id = next(
#         (
#             x.get("accession_number")
#             for x in chebi_request.get("database_accessions", {}).get("MANUAL_X_REF", [])
#             if x.get("prefix") == "hmdb"
#         ),
#         None
#     )

#     ontology_relations = chebi_request["ontology_relations"]
#     ontologies = {
#         "incoming": [
#             f"{rel['init_name']} {rel['relation_type']} {rel['final_name']}" for rel in ontology_relations["incoming_relations"]
#         ],
#         "outgoing": [
#             f"{rel['init_name']} {rel['relation_type']} {rel['final_name']}" for rel in ontology_relations["outgoing_relations"]
#         ]
#     }

#     df.at[id, "HMDB ID"] = hmdb_id
#     df.at[id, "Ontology Links"] = json.dumps(ontologies)

# df.head(5)

## NIST Chemistry WebBook

By InChi, retrieving retention index.

In [ ]:
# for index, row in df.iterrows():
#     inchi = row["InChI"]
#     if pd.isna(inchi):
#         print(f"No InChI for {row['Input Name']}, skipping NIST lookup")
#         continue

nist_compound = nist.get_compound(df["InChI"][0])
print(nist_compound.ID)

nist_compound.get_gas_chromatography()

gc_all = nist_compound.gas_chromat

for gc in gc_all:
    print(gc)
    print(gc.data)
    # print(gc.compound, gc.ri_type, gc.column_type, gc.temp_regime)
    print(gc.data.columns)
    # print(gc.data[['I']])
    # gc.data

# get I where ri_type = "Normal alkane RI" and column_type = "non-polar"